# Olist: exploração dos dados e decisões do RAG

Este notebook registra a exploração **executada sobre os CSVs reais** usados no projeto. Ele explica por que o corpus textual difere do total de pedidos, como os metadados são interpretados e quais conclusões os dados não autorizam.

As saídas são resultados automáticos de células Python executadas em ordem; **não constituem anotação ou auditoria humana**. Não há benchmark de relevância nem avaliação de fidelidade gerada neste notebook. Os experimentos de recuperação e geração ficam em `evaluation/`, com seus próprios limites e procedimentos.

**Reprodução:** execute `python scripts/download_data.py` e depois `python scripts/run_notebook.py`, a partir da raiz do repositório e com o ambiente de `requirements.txt`. O executor usa Python comum em ordem e salva stdout/stderr, sem magics do IPython nem kernel Jupyter. Por padrão usamos `data/raw`; `--raw CAMINHO` permite indicar uma cópia local e `--output CAMINHO.ipynb` preserva o arquivo original. Não há necessidade de conexão a LLM. Também é possível abrir o arquivo num editor de notebooks com kernel do ambiente do projeto.

In [1]:
import csv
import hashlib
import json
import os
import platform
import sys
from collections import Counter
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'voc' / 'data.py').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Execute a partir do repositório ou da pasta notebooks.')
sys.path.insert(0, str(PROJECT_ROOT))
from voc.data import prepare, rows, clean

RAW = Path(os.environ.get('VOC_RAW_DIR', PROJECT_ROOT / 'data' / 'raw'))
required = ['olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv',
            'olist_order_items_dataset.csv', 'olist_products_dataset.csv',
            'olist_customers_dataset.csv']
missing = [name for name in required if not (RAW / name).exists()]
if missing:
    raise FileNotFoundError(f'Baixe os CSVs Olist antes de executar: {missing}')
print('Python:', platform.python_version())
print('Fonte: CSVs Olist locais; cinco tabelas usadas na preparação e nos metadados.')

Python: 3.12.14
Fonte: CSVs Olist locais; cinco tabelas usadas na preparação e nos metadados.


## 1. Proveniência e unidade de análise

O enunciado pede explorar os textos de avaliações. O pedido, o review e o item do pedido são unidades diferentes. Usamos reviews, pedidos, itens e produtos, além da tabela de clientes para obter somente a UF. Não usamos nomes de clientes, geolocalização detalhada nem pagamentos nos joins. O hash identifica os bytes analisados e permite comparar uma nova execução com `evaluation/data-provenance.json`. Ele não atesta, sozinho, a origem: o procedimento de download e a fonte oficial também devem ser considerados.

In [2]:
inventory = []
for name in required:
    path = RAW / name
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    count = sum(1 for _ in rows(path))
    inventory.append({'arquivo': name, 'linhas': count, 'sha256': digest})
print(json.dumps(inventory, ensure_ascii=False, indent=2))

[
  {
    "arquivo": "olist_order_reviews_dataset.csv",
    "linhas": 99224,
    "sha256": "012b61c7593e34f51fa614efdf802b9c7056ce6aae5307ddb93236e7cfc797d7"
  },
  {
    "arquivo": "olist_orders_dataset.csv",
    "linhas": 99441,
    "sha256": "8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf"
  },
  {
    "arquivo": "olist_order_items_dataset.csv",
    "linhas": 112650,
    "sha256": "0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279"
  },
  {
    "arquivo": "olist_products_dataset.csv",
    "linhas": 32951,
    "sha256": "3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc"
  },
  {
    "arquivo": "olist_customers_dataset.csv",
    "linhas": 99441,
    "sha256": "983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2"
  }
]


## 2. Ausência de texto e seleção

Combinamos título e mensagem. Um review sem ambos não oferece conteúdo textual para recuperação; ele é excluído do corpus, embora exista no conjunto original. Não preenchemos textos ausentes com notas, metadados nem texto sintético. Notas fora de 1–5 também são excluídas. Isso cria viés de seleção: resultados descrevem quem deixou texto, não todos os consumidores ou pedidos.

In [3]:
reviews = list(rows(RAW / 'olist_order_reviews_dataset.csv'))
documents, ingestion = prepare(RAW)
raw_missing = {
    'linhas_review': len(reviews),
    'sem_titulo': sum(not (r.get('review_comment_title') or '').strip() for r in reviews),
    'sem_mensagem': sum(not (r.get('review_comment_message') or '').strip() for r in reviews),
    'sem_titulo_e_mensagem': sum(not (r.get('review_comment_title') or '').strip()
                               and not (r.get('review_comment_message') or '').strip() for r in reviews),
}
print('Ausência nos campos originais:')
print(json.dumps(raw_missing, ensure_ascii=False, indent=2))
print('Relatório da preparação, sem limite de amostragem:')
print(json.dumps(ingestion, ensure_ascii=False, indent=2))
assert documents, 'A preparação deve produzir documentos para analisar.'

Ausência nos campos originais:
{
  "linhas_review": 99224,
  "sem_titulo": 87658,
  "sem_mensagem": 58274,
  "sem_titulo_e_mensagem": 56537
}
Relatório da preparação, sem limite de amostragem:
{
  "rows_read": 99224,
  "empty_text": 56537,
  "documents": 42687,
  "score_distribution": {
    "5": 21776,
    "4": 6274,
    "2": 2165,
    "1": 8829,
    "3": 3643
  },
  "unique_textual_reviews": 42370,
  "reviews_with_multiple_orders": 307,
  "delivery_status_distribution_documents": {
    "on_time": 36937,
    "late": 3848,
    "not_delivered": 824,
    "unknown": 1078
  },
  "metadata_schema": 2
}


## 3. Identidade e duplicatas

`review_id` pode aparecer ligado a mais de um pedido. Deduplicar somente por esse campo perderia vínculos; deduplicar por texto removeria relatos iguais de clientes/pedidos diferentes. A preparação usa a combinação review, pedido e texto limpo. O número abaixo é uma conferência de identidade, sem presumir que todo ID repetido seja erro do dataset.

In [4]:
review_orders = {}
for row in reviews:
    review_orders.setdefault(row['review_id'], set()).add(row['order_id'])
identity = {
    'review_ids_distintos_no_csv': len(review_orders),
    'review_ids_em_mais_de_um_pedido_no_csv': sum(len(v) > 1 for v in review_orders.values()),
    'documentos_preparados': len(documents),
    'ids_documento_distintos': len({d['id'] for d in documents}),
    'pedidos_distintos_no_corpus': len({d['order_id'] for d in documents}),
}
print(json.dumps(identity, ensure_ascii=False, indent=2))
assert len({d['id'] for d in documents}) == len(documents)

{
  "review_ids_distintos_no_csv": 98410,
  "review_ids_em_mais_de_um_pedido_no_csv": 789,
  "documentos_preparados": 42687,
  "ids_documento_distintos": 42687,
  "pedidos_distintos_no_corpus": 42540
}


## 4. Notas: satisfação declarada, sem inferir sentimento do texto

Comparar a distribuição original com a textual evidencia a seleção do corpus. A nota é um filtro contextual; não prova o motivo de insatisfação nem substitui um classificador de sentimento validado. As barras abaixo representam contagens e não frequência de temas.

In [5]:
all_scores = Counter(r.get('review_score', '') for r in reviews)
text_scores = Counter(str(d['score']) for d in documents)
print('Nota | Reviews no CSV | Documentos textuais | Parcela textual da nota')
for score in map(str, range(1, 6)):
    denominator = all_scores[score]
    fraction = text_scores[score] / denominator if denominator else 0
    print(f'{score:>4} | {denominator:>14,} | {text_scores[score]:>19,} | {fraction:>21.2%}')
print('\nDocumentos textuais por nota (cada bloco ~= 500 documentos):')
for score in map(str, range(1, 6)):
    print(f'{score}: ' + '#' * round(text_scores[score] / 500) + f' ({text_scores[score]:,})')

Nota | Reviews no CSV | Documentos textuais | Parcela textual da nota
   1 |         11,424 |               8,829 |                77.28%
   2 |          3,151 |               2,165 |                68.71%
   3 |          8,179 |               3,643 |                44.54%
   4 |         19,142 |               6,274 |                32.78%
   5 |         57,328 |              21,776 |                37.98%

Documentos textuais por nota (cada bloco ~= 500 documentos):
1: ################## (8,829)
2: #### (2,165)
3: ####### (3,643)
4: ############# (6,274)
5: ############################################ (21,776)


## 5. Contexto de compra, categorias e entrega

Datas de compra e de review têm significados diferentes; o filtro temporal usa a primeira. As categorias são agregadas por pedido para impedir multiplicação de reviews ao fazer join com itens. Um pedido pode ter várias categorias: contar categorias não equivale a contar documentos e não permite atribuir o comentário a um produto específico.

Entrega posterior à data estimada é um indicador operacional de atraso confirmado. Ausência de data não significa entrega pontual, e uma reclamação de demora pode existir mesmo com entrega dentro da estimativa. Esses conceitos devem aparecer separados na interface e em qualquer análise agregada.

In [6]:
metadata = {
    'sem_data_compra': sum(not d.get('purchase_date') for d in documents),
    'sem_categoria': sum(not d.get('categories') for d in documents),
    'sem_uf_no_vinculo_principal': sum(not d.get('state') for d in documents),
    'documentos_com_multiplas_categorias': sum(len(d.get('categories', [])) > 1 for d in documents),
    'atraso_confirmado': sum(d.get('late_delivery') is True for d in documents),
    'sem_atraso_confirmado_com_datas': sum(d.get('late_delivery') is False for d in documents),
    'atraso_desconhecido_ou_nao_entregue': sum(d.get('late_delivery') is None for d in documents),
}
purchase_dates = sorted(d['purchase_date'] for d in documents if d.get('purchase_date'))
review_dates = sorted(d['review_date'] for d in documents if d.get('review_date'))
print(json.dumps(metadata, ensure_ascii=False, indent=2))
print('Intervalo das compras do corpus:', purchase_dates[0], 'a', purchase_dates[-1])
print('Intervalo das avaliações do corpus:', review_dates[0], 'a', review_dates[-1])
if any('delivery_status' in d for d in documents):
    print('Estados operacionais de entrega:', dict(Counter(d.get('delivery_status', 'ausente') for d in documents)))
category_counts = Counter(c for d in documents for c in d.get('categories', []))
print('Categorias mais presentes nos vínculos de documentos:', category_counts.most_common(5))
print('As contagens de categorias podem se sobrepor; não somá-las como total de reviews.')

{
  "sem_data_compra": 0,
  "sem_categoria": 1133,
  "sem_uf_no_vinculo_principal": 0,
  "documentos_com_multiplas_categorias": 464,
  "atraso_confirmado": 3848,
  "sem_atraso_confirmado_com_datas": 36937,
  "atraso_desconhecido_ou_nao_entregue": 1902
}
Intervalo das compras do corpus: 2016-09-04 a 2018-10-17
Intervalo das avaliações do corpus: 2016-10-02 a 2018-08-31
Estados operacionais de entrega: {'on_time': 36937, 'late': 3848, 'not_delivered': 824, 'unknown': 1078}
Categorias mais presentes nos vínculos de documentos: [('cama_mesa_banho', 4471), ('beleza_saude', 3588), ('esporte_lazer', 3087), ('informatica_acessorios', 2839), ('moveis_decoracao', 2695)]
As contagens de categorias podem se sobrepor; não somá-las como total de reviews.


## 6. Limpeza e tamanho dos documentos

Preservamos acentos, negações e erros linguísticos porque afetam a interpretação. HTML e espaços excedentes são removidos. O mascaramento por padrões é parcial e não equivale a anonimização: nomes, endereços e outros dados pessoais ainda podem ocorrer. Os exemplos abaixo são artificiais e servem somente para ilustrar a função de limpeza, sem expor um consumidor real.

In [7]:
examples = [
    '<p>Não gostei</p>   do produto.',
    'Dúvida: pessoa.exemplo@example.com; telefone (11) 98765-4321',
    'Entrega atrasou, mas o produto é ótimo.',
]
for example in examples:
    print(repr(example), '=>', repr(clean(example)))
lengths = sorted(len(d['text'].split()) for d in documents)
def empirical_percentile(sorted_values, p):
    return sorted_values[round((len(sorted_values) - 1) * p)]
print('Tamanho em palavras:', {
    'minimo': lengths[0], 'mediana': empirical_percentile(lengths, 0.5),
    'p95': empirical_percentile(lengths, 0.95), 'maximo': lengths[-1],
    'acima_90_palavras': sum(n > 90 for n in lengths),
})
print('Palavras não equivalem a tokens; estes números não garantem ausência de truncamento pelo tokenizer.')

'<p>Não gostei</p>   do produto.' => 'Não gostei do produto.'
'Dúvida: pessoa.exemplo@example.com; telefone (11) 98765-4321' => 'Dúvida: [EMAIL]; telefone [TELEFONE]'
'Entrega atrasou, mas o produto é ótimo.' => 'Entrega atrasou, mas o produto é ótimo.'
Tamanho em palavras: {'minimo': 1, 'mediana': 9, 'p95': 33, 'maximo': 48, 'acima_90_palavras': 0}
Palavras não equivalem a tokens; estes números não garantem ausência de truncamento pelo tokenizer.


## 7. Conferência do índice existente

O modelo multilíngue foi escolhido para textos em português e a busca exata NumPy evita um serviço adicional no porte deste protótipo. Vetores normalizados permitem cosseno por produto escalar. O índice registra a configuração do encoder para que consulta e corpus usem a mesma variante.

A célula seguinte **não reconstrói nem modifica o índice**. Se `data/index` estiver ausente, informa o comando de ingestão. Um índice existente pode pertencer a uma versão anterior: confronte seu manifesto, o código e os artefatos de avaliação antes de usar os resultados. Contagens, norma e dimensão são verificações estruturais, não métricas de relevância.

In [8]:
import numpy as np

INDEX = PROJECT_ROOT / 'data' / 'index'
if (INDEX / 'manifest.json').exists() and (INDEX / 'vectors.npy').exists():
    manifest = json.loads((INDEX / 'manifest.json').read_text(encoding='utf-8'))
    vectors = np.load(INDEX / 'vectors.npy', mmap_mode='r', allow_pickle=False)
    norms = np.linalg.norm(vectors, axis=1)
    summary = {
        'modelo': manifest.get('model'), 'quantized': manifest.get('quantized'),
        'schema': manifest.get('schema'), 'shape': list(vectors.shape),
        'dtype': str(vectors.dtype), 'todos_finitos': bool(np.isfinite(vectors).all()),
        'norma_minima': float(norms.min()), 'norma_maxima': float(norms.max()),
        'fragmentos_no_manifesto': manifest.get('chunks'),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    assert np.isfinite(vectors).all()
    assert len(vectors) == manifest['chunks']
else:
    print('Índice ausente. Execute: python -m voc.cli ingest --raw data/raw')

{
  "modelo": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "quantized": true,
  "schema": 2,
  "shape": [
    42687,
    384
  ],
  "dtype": "float32",
  "todos_finitos": true,
  "norma_minima": 0.9999998211860657,
  "norma_maxima": 1.0000001192092896,
  "fragmentos_no_manifesto": 42687
}


## 8. Decisões e limites para interpretação

| Decisão | Razão | Limite |
|---|---|---|
| Corpus apenas textual | Recuperar falas reais em vez de fabricar comentários ausentes | Não representa todos os clientes |
| Identidade review + pedido + texto | Preservar vínculos de reviews repetidos | Textos iguais podem permanecer legitimamente |
| Categorias agregadas por pedido | Evitar multiplicar reviews nos joins | Não atribui a fala a um item específico |
| Embeddings multilíngues e cosseno | Explorar paráfrases em português | Similaridade não é probabilidade de relevância |
| Recuperação densa + BM25 e RRF, sem MMR no padrão | Combinar semântica e termos específicos; configuração escolhida no desenvolvimento | Benchmark silver limitado; top-k não é amostra aleatória |
| Filtros antes do ranking e metadados por pedido | Respeitar o contexto de compra e preservar vínculos | Um review ligado a vários pedidos exige unidade de contagem explícita |
| Evidências citadas | Permitir voltar ao comentário de origem | Citação existente não prova apoio semântico completo |
| Síntese restrita e validação de trechos | Reduzir espaço para afirmações livres sem suporte | Restringe expressividade; não elimina todo erro de interpretação |
| Análise agregada separada do top-k | Exibir contagens com denominador e regra explícitos | Regras lexicais são proxies e podem errar ou se sobrepor |

Perguntas como “quais problemas são mais frequentes?” exigem cálculo sobre o corpus filtrado, definição de tema e denominador, sem generalizar uma busca top-k. Causalidade, preços atuais, políticas internas e fatos ausentes devem produzir recusa. A nota baixa não comprova uma causa; atraso operacional não é sinônimo de relato de demora.

As avaliações automáticas do projeto devem ser lidas junto com `docs/avaliacao.md`, `docs/criterios-aceite.md` e os arquivos em `evaluation/`. Não há afirmação de benchmark humano, desempenho universal ou execução de Docker neste notebook. Para comprovar geração RAG, use os experimentos reais de LLM e a interface: somente recuperar comentários em modo extrativo não cumpre a etapa de geração.

In [9]:
artifacts = [
    'evaluation/data-provenance.json', 'evaluation/results.json',
    'evaluation/llm-demo.json', 'evaluation/validation.md',
    'docs/criterios-aceite.md', 'docs/avaliacao.md',
]
print('Artefatos para inspeção complementar (presença não equivale a aprovação):')
for artifact in artifacts:
    print(artifact, '=>', 'presente' if (PROJECT_ROOT / artifact).exists() else 'ausente')
print('Este notebook não executa nem substitui os testes, a avaliação de recuperação ou a demonstração LLM.')

Artefatos para inspeção complementar (presença não equivale a aprovação):
evaluation/data-provenance.json => presente
evaluation/results.json => presente
evaluation/llm-demo.json => presente
evaluation/validation.md => presente
docs/criterios-aceite.md => presente
docs/avaliacao.md => presente
Este notebook não executa nem substitui os testes, a avaliação de recuperação ou a demonstração LLM.
